[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imbishal7/CSC-428/blob/main/notebooks/CSC428_project.ipynb)

# CSC428 Final Project — ML Side-Channel Attack on AES (ASCAD)

Suwan Aryal & Kapil Sharma. Single-trace classification: from one 700-sample power trace recorded during a masked-AES execution, predict the Hamming weight (0..8) of the masked Sbox output.

**Pipeline:**

1. Data downloading and loading — build one DataFrame with 700 trace columns + `label`
2. Data cleaning — NaN check, label engineering (auto-detect mask byte by SNR), standardization
3. Train/test split via **5-fold stratified cross-validation**
4. Define, train, and test three models — Logistic Regression, MLP, CNN — under the same CV folds
5. Compare results (mean ± std across folds)

## Section 0 — Colab setup
Mount Drive, clone the repo (first run only), install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
![ -d CSC-428 ] || git clone https://github.com/imbishal7/CSC-428.git
%cd CSC-428
!pip install -q -r requirements.txt
!nvidia-smi -L

## 1. Data downloading and loading

ANSSI publishes ASCAD as a single ~4.2 GB zip. The download script grabs it and extracts `ASCAD.h5`, idempotently.

We then build **one combined DataFrame** with every available trace (50,000 profiling + 10,000 attack = 60,000 rows). Columns:

- `t_0, t_1, ..., t_699` — the 700 oscilloscope samples per trace (the raw features).
- `label` — integer 0..8 we'll predict (filled in section 2 from the unmasked Sbox value and the relevant mask byte).

We keep the unmasked Sbox value and the per-trace masks in temporary numpy arrays just long enough to derive `label`; once `label` is computed they're not part of the DataFrame so it stays clean (features + target only).

In [ ]:
!python scripts/download_data.py

In [ ]:
import h5py
import numpy as np
import pandas as pd

HW_TABLE = np.array([bin(i).count('1') for i in range(256)], dtype=np.int64)

with h5py.File('data/ASCAD.h5', 'r') as f:
    traces = np.concatenate([
        np.array(f['Profiling_traces/traces'], dtype=np.float32),
        np.array(f['Attack_traces/traces'],     dtype=np.float32),
    ], axis=0)
    sbox_unmasked = np.concatenate([
        np.array(f['Profiling_traces/labels'], dtype=np.int64),
        np.array(f['Attack_traces/labels'],     dtype=np.int64),
    ], axis=0)
    masks = np.concatenate([
        np.array(f['Profiling_traces/metadata']['masks'], dtype=np.int64),
        np.array(f['Attack_traces/metadata']['masks'],     dtype=np.int64),
    ], axis=0)

n, n_samples = traces.shape
print(f'combined: {n} traces  x  {n_samples} samples each')
print(f'mask bytes per trace: {masks.shape[1]}')

feature_cols = [f't_{i}' for i in range(n_samples)]
df = pd.DataFrame(traces, columns=feature_cols)
df['label'] = -1   # placeholder; computed in section 2 once the right mask byte is picked

print(f'\ndf shape: {df.shape}')
print('df.head() (first 5 trace columns + label):')
df[feature_cols[:5] + ['label']].head()

In [ ]:
df.info(memory_usage='deep')

In [ ]:
# Statistical summary across a representative slice of trace columns (every 100th)
df[feature_cols[::100]].describe()

## 2. Data cleaning + label engineering

**(a) Sanity checks.** No NaN/Inf in the trace columns.

**(b) Pick the right label.** ASCAD's chip runs *boolean-masked* AES, so the value the trace physically leaks at the Sbox computation is `Sbox(p ^ k) ^ mask_out` — not `Sbox(p ^ k)`. We auto-detect which mask byte is the relevant `mask_out` by computing per-sample SNR for every candidate `HW(Sbox(p^k) ^ masks[:, m])` and picking the column with the highest peak. The unmasked label has SNR ≈ 0 (the mask cryptographically hides it from any single trace), so the difference is unmistakable.

**(c) Standardize the trace columns** (zero mean, unit variance per sample) so all models train on the same scale.

> **Caveat for the report.** Using the masks at training time is a *profiling-attack* assumption: the attacker has full control of a clone device and can read its masks. At deployment, recovering the unmasked value from masks-unknown traces requires higher-order analysis. We're showing what's learnable in principle from one trace once the right label is identified.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
Path('results').mkdir(exist_ok=True)

# (a) sanity
trace_block = df[feature_cols].values
assert not np.isnan(trace_block).any() and not np.isinf(trace_block).any(), 'NaN/Inf in trace columns'
print('NaN per trace column (top 5):', df.isna().sum().sort_values(ascending=False).head().to_dict())
print('no NaN / Inf')

def per_sample_snr(traces, labels, n_classes=9):
    means = np.zeros((n_classes, traces.shape[1]), dtype=np.float64)
    var_w = np.zeros_like(means)
    counts = np.zeros(n_classes, dtype=np.int64)
    for c in range(n_classes):
        sl = labels == c
        n = int(sl.sum())
        counts[c] = n
        if n == 0:
            continue
        sub = traces[sl].astype(np.float64)
        means[c] = sub.mean(axis=0)
        var_w[c] = sub.var(axis=0)
    grand = (means * counts[:, None]).sum(0) / counts.sum()
    w = counts[:, None] / counts.sum()
    var_b = ((means - grand) ** 2 * w).sum(0)
    return var_b / np.maximum((var_w * w).sum(0), 1e-12)

# (b) auto-detect best mask byte using a 10k slice (faster than the full 60k)
N_DIAG = min(10000, len(df))
snr_per_mask = []
for m_idx in range(masks.shape[1]):
    cand = HW_TABLE[(sbox_unmasked[:N_DIAG] ^ masks[:N_DIAG, m_idx]) & 0xff]
    snr_per_mask.append(per_sample_snr(trace_block[:N_DIAG], cand).max())
snr_per_mask = np.array(snr_per_mask)
best_mask = int(snr_per_mask.argmax())
snr_unmasked = per_sample_snr(trace_block[:N_DIAG], HW_TABLE[sbox_unmasked[:N_DIAG]])

print(f'\nper-mask peak SNR: {snr_per_mask.round(4).tolist()}')
print(f'best mask byte: index {best_mask}  (SNR {snr_per_mask[best_mask]:.3f}  vs unmasked SNR {snr_unmasked.max():.4f})')
assert snr_per_mask[best_mask] > 10 * snr_unmasked.max(), 'no mask byte clearly dominates'

df['label'] = HW_TABLE[(sbox_unmasked ^ masks[:, best_mask]) & 0xff]
print(f"\nlabel value_counts:\n{df['label'].value_counts().sort_index().to_string()}")

snr_best = per_sample_snr(trace_block[:N_DIAG], df['label'].values[:N_DIAG])
fig, ax = plt.subplots(2, 2, figsize=(13, 6))
ax[0, 0].plot(snr_unmasked);                ax[0, 0].set(title='SNR with HW(Sbox(p^k)) -- masked away',           xlabel='time sample', ylabel='SNR')
ax[0, 1].plot(snr_best, color='tab:green'); ax[0, 1].set(title=f'SNR with HW(Sbox(p^k) ^ masks[:,{best_mask}])',  xlabel='time sample', ylabel='SNR')
df['label'].value_counts().sort_index().plot(kind='bar', ax=ax[1, 0]); ax[1, 0].set(title='Label distribution', xlabel='Hamming weight', ylabel='count')
ax[1, 1].plot(trace_block[0]); ax[1, 1].set(title='Example trace (row 0)', xlabel='time sample', ylabel='power')
for row in ax:
    for a in row: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/snr_and_distribution.png', dpi=120, bbox_inches='tight'); plt.show()

# (c) standardize trace columns in place
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(trace_block)
df.loc[:, feature_cols] = scaler.transform(trace_block).astype(np.float32)
print('\nstandardized -- df.head() (first 5 trace cols + label):')
df[feature_cols[:5] + ['label']].head()

## 3. Train/test split via stratified 5-fold cross-validation

Replace the single train/val/test partition with **5-fold stratified CV**: shuffle the 60,000 rows, split into 5 equal folds preserving class proportions, train each model on 4 folds and evaluate on the held-out 5th, rotating five times. The reported test accuracy is the mean across folds with std as an uncertainty bar.

Inside each training fold we carve a small slice for `validation_data` so EarlyStopping has something to monitor.

In [ ]:
from sklearn.model_selection import StratifiedKFold

X_all = df[feature_cols].values.astype(np.float32)
y_all = df['label'].values.astype(np.int64)
print(f'X_all: {X_all.shape}   y_all: {y_all.shape}')

K = 5
kfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
for i, (tr, te) in enumerate(kfold.split(X_all, y_all)):
    print(f'  fold {i+1}: train={len(tr):>5d}   test={len(te):>5d}')

## 4. Models — define, train, test under k-fold CV

Three models, increasing capacity:

- **Logistic Regression** — one Dense softmax layer.
- **MLP** — two hidden Dense(256) layers + Dropout(0.3).
- **CNN** — two Conv1D + AvgPool blocks + Flatten + Dense(128) + Dropout(0.5).

All trained with sparse categorical cross-entropy. Logreg uses Adam @ 1e-3; the deeper models use RMSprop @ 1e-4 (Adam @ 1e-3 was too aggressive — drove training accuracy to 100% with diverging val loss). EarlyStopping on `val_loss` (patience 15, restore best) prevents memorization.

> **Reference baseline.** HW labels follow `Binom(8, 0.5)`; HW=4 alone is `C(8,4)/256 ≈ 27.3%` of the data, so always-predict-4 already scores ~0.27 test accuracy. Real models must clear that decisively.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

INPUT_LEN = X_all.shape[1]
N_CLASSES = 9
EPOCHS    = 100
BATCH     = 256

def early_stop():
    return keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=0,
    )

def build_logreg():
    m = keras.Sequential([
        layers.Input(shape=(INPUT_LEN,)),
        layers.Dense(N_CLASSES, activation='softmax'),
    ], name='logreg')
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_mlp():
    m = keras.Sequential([
        layers.Input(shape=(INPUT_LEN,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(N_CLASSES, activation='softmax'),
    ], name='mlp')
    m.compile(optimizer=keras.optimizers.RMSprop(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_cnn():
    m = keras.Sequential([
        layers.Input(shape=(INPUT_LEN, 1)),
        layers.Conv1D(32, 11, activation='relu', padding='same'),
        layers.AveragePooling1D(2),
        layers.Conv1D(64, 11, activation='relu', padding='same'),
        layers.AveragePooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(N_CLASSES, activation='softmax'),
    ], name='cnn')
    m.compile(optimizer=keras.optimizers.RMSprop(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def kfold_cv(builder, X, y, channel=False):
    """Run K-fold CV; return per-fold test accuracies."""
    accs = []
    for fold_idx, (tr_idx, te_idx) in enumerate(kfold.split(X, y)):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        n_val = max(1, len(X_tr) // 9)   # ~10% of train fold for early-stopping val
        X_inner_tr, X_val = X_tr[:-n_val], X_tr[-n_val:]
        y_inner_tr, y_val = y_tr[:-n_val], y_tr[-n_val:]
        if channel:
            X_inner_tr, X_val, X_te = X_inner_tr[..., None], X_val[..., None], X_te[..., None]
        model = builder()
        model.fit(X_inner_tr, y_inner_tr, validation_data=(X_val, y_val),
                  epochs=EPOCHS, batch_size=BATCH, verbose=0,
                  callbacks=[early_stop()])
        _, acc = model.evaluate(X_te, y_te, batch_size=512, verbose=0)
        accs.append(acc)
        print(f'  fold {fold_idx+1}/{K}: test_acc = {acc:.4f}')
    return np.array(accs)

results = {}
majority_class = int(np.bincount(y_all, minlength=9).argmax())
results['Majority-class baseline'] = np.full(K, float((y_all == majority_class).mean()))
print(f"majority-class baseline: {results['Majority-class baseline'][0]:.4f}")

### 4a. Logistic Regression

In [ ]:
print('=== Logistic Regression ===')
results['Logistic Regression'] = kfold_cv(build_logreg, X_all, y_all, channel=False)
print(f"  mean test_acc = {results['Logistic Regression'].mean():.4f}  +/-  {results['Logistic Regression'].std():.4f}")

### 4b. MLP

In [ ]:
print('=== MLP ===')
results['MLP'] = kfold_cv(build_mlp, X_all, y_all, channel=False)
print(f"  mean test_acc = {results['MLP'].mean():.4f}  +/-  {results['MLP'].std():.4f}")

### 4c. CNN

In [ ]:
print('=== CNN ===')
results['CNN'] = kfold_cv(build_cnn, X_all, y_all, channel=True)
print(f"  mean test_acc = {results['CNN'].mean():.4f}  +/-  {results['CNN'].std():.4f}")

## 5. Results comparison

Mean ± std test accuracy across the 5 folds, plus per-fold accuracies for inspection.

In [ ]:
summary = pd.DataFrame({name: arr for name, arr in results.items()},
                       index=[f'fold_{i+1}' for i in range(K)])
summary.loc['mean'] = summary.iloc[:K].mean()
summary.loc['std']  = summary.iloc[:K].std()
summary.to_csv('results/cv_results.csv')
print(summary.round(4).to_string())

means = summary.loc['mean']
stds  = summary.loc['std']
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(means.index, means.values, yerr=stds.values, capsize=5,
       color=['#aaaaaa', '#1f77b4', '#ff7f0e', '#2ca02c'])
for i, (m, s) in enumerate(zip(means.values, stds.values)):
    ax.text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontsize=9)
ax.set_ylabel('Test accuracy (5-fold mean ± std)')
ax.set_title('Model comparison — single-trace HW classification (k-fold CV)')
ax.set_ylim(0, max(means.values) * 1.2)
ax.grid(axis='y', alpha=0.3); plt.xticks(rotation=15); plt.tight_layout()
plt.savefig('results/comparison.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for name in [n for n in results if n != 'Majority-class baseline']:
    ax.plot(range(1, K+1), results[name], marker='o', label=name)
ax.axhline(results['Majority-class baseline'][0], color='gray', linestyle='--', label='Majority baseline')
ax.set_xlabel('fold'); ax.set_ylabel('test accuracy')
ax.grid(alpha=0.3); ax.legend(); ax.set_xticks(range(1, K+1))
plt.title('Per-fold test accuracy'); plt.tight_layout()
plt.savefig('results/per_fold.png', dpi=120, bbox_inches='tight'); plt.show()